In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp, lit

# 1. Extração dos dados da API OpenAlex
QUERY_TOPIC = "artificial intelligence"
MAILTO_EMAIL = "depaulasilvadiego297@gmail.com"
url = f"https://api.openalex.org/works?filter=default.search:{QUERY_TOPIC}&per-page=50&mailto={MAILTO_EMAIL}"
o
print(f"Iniciando requisição para: {url}")
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    records = data.get("results", [])
    print(f"Sucesso: {len(records)} registros brutos recuperados.")
else:
    raise Exception(f"Erro na API OpenAlex: Status {response.status_code} - {response.text}")

# 2. Higienização: descarta o campo problemático antes da inferência do Spark
for item in records:
    item.pop("abstract_inverted_index", None)

# 3. Provisionamento de Volume no Unity Catalog
catalogo = spark.sql("SELECT current_catalog()").collect()[0][0]
esquema = spark.sql("SELECT current_database()").collect()[0][0]
volume_nome = "bronze_volume"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.{esquema}.{volume_nome}")
caminho_raw = f"/Volumes/{catalogo}/{esquema}/{volume_nome}/raw_openalex.json"

# 4. Gravação física do JSON no Volume
with open(caminho_raw, "w", encoding="utf-8") as f:
    json.dump(records, f)

print(f"Arquivo persistido com sucesso em: {caminho_raw}")

# 5. Leitura PySpark direta (sem necessidade de flags de case sensitivity)
df_bronze = spark.read.json(caminho_raw)

# 6. Adição de metadados de auditoria e gravação em Delta Lake
df_bronze_final = df_bronze \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system", lit("OpenAlex API"))

tabela_bronze = "bronze_openalex"
df_bronze_final.write.format("delta").mode("overwrite").saveAsTable(tabela_bronze)

print(f"Tabela Delta '{tabela_bronze}' criada com sucesso!")
display(spark.table(tabela_bronze).limit(5))